In [1]:
!pip install groq gradio Pillow opencv-python-headless -q
print("✅ All installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.6 MB/s eta 0:00:00
✅ All installed


In [2]:
"""
CELL 2 — Backend Engine
================================
run it before launching the UI.
"""

import gradio as gr
import base64, json, re, os
from groq import Groq
from PIL import Image
import io

GROQ_API_KEY = "ENTER YOUR API KEY HEREEEE!!!!!!!!!!!!!!!"
client = Groq(api_key=GROQ_API_KEY)

def image_to_base64(pil_img):
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def extract_text_from_images(images):
    all_text = []
    for i, img in enumerate(images):
        if img is None:
            continue
        b64 = image_to_base64(img)
        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{"role": "user", "content": [
                {"type": "text", "text": """You are an expert OCR for handwritten exam answer sheets.
Extract ALL handwritten text. Label each answer as Q1, Q2, Q3 etc.
Output ONLY the extracted text in this format:
Q1: [full answer]
Q2: [full answer]
Do NOT add your own content."""},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}
            ]}],
            max_tokens=3000, temperature=0.1
        )
        all_text.append(f"--- Page {i+1} ---\n{response.choices[0].message.content}")
    return "\n\n".join(all_text)

def grade_answers(extracted_text, mode, answer_key_text, policy):
    # Policy is now respected in BOTH modes
    policy_instruction = f"Apply a {policy.lower()} grading approach: " + {
        "Strict": "award marks only for precise, complete answers. Deduct for vagueness, partial answers, or minor errors.",
        "Moderate": "balance correctness with understanding. Give partial credit for mostly correct answers.",
        "Lenient": "reward demonstrated understanding. Give generous partial credit and focus on core concepts."
    }.get(policy, "balance correctness with understanding.")

    if mode == "Auto (No Answer Key)":
        prompt = f"""You are an autonomous exam grader.

STUDENT ANSWERS:
{extracted_text}

GRADING POLICY: {policy_instruction}

TASK: Identify each question and evaluate the answer out of 5 marks based on factual correctness and completeness.

Respond ONLY in valid JSON:
{{
  "Q1": {{"marks_awarded": 4, "max_marks": 5, "strengths": "...", "weaknesses": "...", "feedback": "..."}},
  "Q2": {{"marks_awarded": 3, "max_marks": 5, "strengths": "...", "weaknesses": "...", "feedback": "..."}},
  "overall_comment": "...",
  "total_marks": 7,
  "total_max": 10,
  "percentage": 70.0
}}"""
    else:
        prompt = f"""You are a strict exam grader.

STUDENT ANSWERS:
{extracted_text}

ANSWER KEY:
{answer_key_text}

GRADING POLICY: {policy_instruction}

Grade each question. Respond ONLY in valid JSON:
{{
  "Q1": {{"marks_awarded": 4, "max_marks": 5, "strengths": "...", "weaknesses": "...", "feedback": "..."}},
  "overall_comment": "...",
  "total_marks": 4,
  "total_max": 5,
  "percentage": 80.0
}}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Output strictly valid JSON only. No markdown, no backticks."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.1, max_tokens=4000
    )
    return json.loads(response.choices[0].message.content)

def format_results(result):
    pct = result.get('percentage', 0)
    grade = "A" if pct >= 90 else "B" if pct >= 75 else "C" if pct >= 60 else "D" if pct >= 50 else "F"
    grade_color = {"A": "#22c55e", "B": "#84cc16", "C": "#f59e0b", "D": "#f97316", "F": "#ef4444"}[grade]

    out = f"""<div style="font-family: 'DM Sans', system-ui, sans-serif; max-width: 720px; margin: 0 auto;">

<div style="display: flex; align-items: center; gap: 24px; padding: 24px; background: var(--bg-card); border: 1px solid var(--border); border-radius: 16px; margin-bottom: 20px;">
  <div style="text-align: center; min-width: 80px;">
    <div style="font-size: 48px; font-weight: 700; color: {grade_color}; line-height: 1;">{grade}</div>
    <div style="font-size: 12px; color: var(--text-muted); margin-top: 2px;">GRADE</div>
  </div>
  <div style="flex: 1; border-left: 1px solid var(--border); padding-left: 24px;">
    <div style="font-size: 28px; font-weight: 600; color: var(--text);">{result.get('total_marks', 0):.1f} <span style="font-size: 16px; color: var(--text-muted); font-weight: 400;">/ {result.get('total_max', 0)} marks</span></div>
    <div style="margin-top: 6px; background: var(--bg-track); border-radius: 99px; height: 8px; overflow: hidden;">
      <div style="width: {pct}%; height: 100%; background: {grade_color}; border-radius: 99px; transition: width 0.6s ease;"></div>
    </div>
    <div style="font-size: 13px; color: var(--text-muted); margin-top: 6px;">{pct:.1f}% — {result.get('overall_comment', '')}</div>
  </div>
</div>

"""

    for key in sorted([k for k in result if re.match(r'^Q\d+$', k)]):
        q = result[key]
        marks = q.get('marks_awarded', 0)
        max_m = q.get('max_marks', 5)
        ratio = marks / max_m if max_m else 0
        bar_color = "#22c55e" if ratio >= 0.7 else ("#f59e0b" if ratio >= 0.5 else "#ef4444")
        status_label = "Strong" if ratio >= 0.7 else ("Partial" if ratio >= 0.5 else "Needs work")

        out += f"""<div style="background: var(--bg-card); border: 1px solid var(--border); border-radius: 12px; padding: 18px 20px; margin-bottom: 12px;">
  <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
    <span style="font-weight: 600; font-size: 15px; color: var(--text);">{key}</span>
    <div style="display: flex; align-items: center; gap: 10px;">
      <span style="font-size: 12px; padding: 2px 10px; border-radius: 99px; background: {bar_color}22; color: {bar_color}; font-weight: 500;">{status_label}</span>
      <span style="font-size: 15px; font-weight: 600; color: var(--text);">{marks} <span style="color: var(--text-muted); font-weight: 400; font-size: 13px;">/ {max_m}</span></span>
    </div>
  </div>
  <div style="background: var(--bg-track); border-radius: 99px; height: 5px; margin-bottom: 14px; overflow: hidden;">
    <div style="width: {ratio*100:.0f}%; height: 100%; background: {bar_color}; border-radius: 99px;"></div>
  </div>
  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-bottom: 10px;">
    <div style="background: #22c55e11; border: 1px solid #22c55e33; border-radius: 8px; padding: 10px 12px;">
      <div style="font-size: 11px; font-weight: 600; color: #16a34a; letter-spacing: 0.05em; margin-bottom: 4px;">STRENGTHS</div>
      <div style="font-size: 13px; color: var(--text);">{q.get('strengths', '—')}</div>
    </div>
    <div style="background: #f59e0b11; border: 1px solid #f59e0b33; border-radius: 8px; padding: 10px 12px;">
      <div style="font-size: 11px; font-weight: 600; color: #d97706; letter-spacing: 0.05em; margin-bottom: 4px;">AREAS TO IMPROVE</div>
      <div style="font-size: 13px; color: var(--text);">{q.get('weaknesses', '—')}</div>
    </div>
  </div>
  <div style="font-size: 13px; color: var(--text-muted); border-top: 1px solid var(--border); padding-top: 10px;">
    <span style="font-weight: 500; color: var(--text);">Feedback:</span> {q.get('feedback', '—')}
  </div>
</div>
"""

    out += "</div>"
    return out

def run_evaluation(img1, img2, img3, img4, mode, answer_key_text, policy):
    images = [i for i in [img1, img2, img3, img4] if i is not None]
    if not images:
        return "❌ Please upload at least one answer sheet image.", ""

    try:
        status = "🔍 Extracting handwritten text via Llama 4 Scout Vision..."
        yield status, ""

        extracted = extract_text_from_images(images)

        status = f"🧠 Grading with Llama 3.3 70B ({policy} policy)..."
        yield status, f"**Extracted Text:**\n```\n{extracted}\n```"

        result = grade_answers(extracted, mode, answer_key_text, policy)
        formatted = format_results(result)

        yield "✅ Evaluation complete", formatted

    except Exception as e:
        yield f"❌ Error: {str(e)}", ""

print("✅ Engine ready")

✅ Engine ready


In [3]:
"""
CELL 3 — UI (run after Cell 2)
================================
"""

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&family=DM+Mono&display=swap');

:root {
    --bg: #f4f3ef;
    --bg-card: #ffffff;
    --bg-track: #e5e7eb;
    --text: #111827;
    --text-muted: #6b7280;
    --border: #e5e7eb;
    --accent: #4f46e5;
    --accent-soft: #eef2ff;
    --radius: 14px;
}

.dark {
    --bg: #0f0f11;
    --bg-card: #1a1a1f;
    --bg-track: #2a2a32;
    --text: #f1f0ed;
    --text-muted: #9ca3af;
    --border: #2a2a32;
    --accent: #818cf8;
    --accent-soft: #1e1b4b;
}

body, .gradio-container {
    background: var(--bg) !important;
    font-family: 'DM Sans', system-ui, sans-serif !important;
    transition: background 0.3s ease;
}

/* App header */
.app-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 20px 0 28px;
    border-bottom: 1px solid var(--border);
    margin-bottom: 28px;
}

.app-title {
    font-size: 22px;
    font-weight: 600;
    color: var(--text);
    letter-spacing: -0.02em;
}

.app-subtitle {
    font-size: 13px;
    color: var(--text-muted);
    margin-top: 2px;
}

/* Cards */
.panel-card {
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 20px;
    margin-bottom: 16px;
    transition: background 0.3s, border 0.3s;
}

.section-label {
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 0.08em;
    color: var(--text-muted);
    text-transform: uppercase;
    margin-bottom: 12px;
}

/* Image upload grid */
.upload-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 10px;
}

/* Gradio overrides */
.gradio-container .gr-box,
.gradio-container .gr-panel {
    background: var(--bg-card) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
}

label span {
    font-size: 12px !important;
    font-weight: 500 !important;
    color: var(--text-muted) !important;
    font-family: 'DM Sans', system-ui, sans-serif !important;
}

.gr-button-primary {
    background: var(--accent) !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    font-family: 'DM Sans', system-ui, sans-serif !important;
    padding: 14px 24px !important;
    width: 100% !important;
    transition: opacity 0.2s !important;
}

.gr-button-primary:hover { opacity: 0.88 !important; }

.gr-radio-row label, .gr-dropdown label {
    font-size: 13px !important;
    color: var(--text) !important;
}

textarea, input[type="text"] {
    font-family: 'DM Mono', monospace !important;
    font-size: 13px !important;
    background: var(--bg) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    color: var(--text) !important;
    padding: 10px 12px !important;
}

/* Status box */
.status-box textarea {
    font-family: 'DM Sans', system-ui, sans-serif !important;
    font-size: 14px !important;
    font-weight: 500 !important;
    color: var(--text) !important;
    background: var(--bg-card) !important;
}

/* Mode toggle pill */
.gr-radio {
    display: flex !important;
    gap: 6px !important;
    background: var(--bg) !important;
    border-radius: 10px !important;
    padding: 4px !important;
    border: 1px solid var(--border) !important;
}

/* Dark mode toggle button */
#dark-toggle {
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: 8px;
    color: var(--text);
    padding: 6px 14px;
    font-size: 13px;
    cursor: pointer;
    font-family: 'DM Sans', system-ui, sans-serif;
    transition: background 0.2s;
}

#dark-toggle:hover { background: var(--accent-soft); }

/* Result markdown */
.result-area {
    --bg-card: var(--bg-card);
    --border: var(--border);
    --text: var(--text);
    --text-muted: var(--text-muted);
    --bg-track: var(--bg-track);
}

/* Scrollbar */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 99px; }
"""

DARK_TOGGLE_JS = """
function() {
    const root = document.documentElement;
    root.classList.toggle('dark');
    return root.classList.contains('dark') ? '☀ Light mode' : '☾ Dark mode';
}
"""

with gr.Blocks(css=CUSTOM_CSS, title="ExamAI — Answer Sheet Evaluator") as demo:

    # ── Header ──
    gr.HTML("""
    <div class="app-header">
      <div>
        <div class="app-title">ExamAI</div>
        <div class="app-subtitle">Handwritten answer sheet evaluator powered by Llama 4 Vision + Llama 3.3 70B</div>
      </div>
    </div>
    """)

    with gr.Row(equal_height=False):

        # ── LEFT PANEL ──
        with gr.Column(scale=4, min_width=340):

            gr.HTML('<div class="section-label">Answer sheets</div>')

            with gr.Row():
                img1 = gr.Image(label="Page 1", type="pil", height=180)
                img2 = gr.Image(label="Page 2", type="pil", height=180)
            with gr.Row():
                img3 = gr.Image(label="Page 3 (opt)", type="pil", height=180)
                img4 = gr.Image(label="Page 4 (opt)", type="pil", height=180)

            gr.HTML('<div class="section-label" style="margin-top:20px;">Grading settings</div>')

            mode = gr.Radio(
                ["Auto (No Answer Key)", "Custom (With Answer Key)"],
                label="Mode",
                value="Auto (No Answer Key)"
            )

            # Policy is ALWAYS visible — applies in both modes
            policy = gr.Dropdown(
                choices=["Strict", "Moderate", "Lenient"],
                label="Grading strictness",
                value="Moderate",
                info="Applies in both Auto and Custom modes"
            )

            answer_key = gr.Textbox(
                label="Answer key",
                placeholder="Q1: Data independence means the separation of data...\nQ2: Integrity constraints ensure...",
                lines=6,
                visible=False
            )

            def toggle_key(m):
                return gr.update(visible=(m == "Custom (With Answer Key)"))

            mode.change(toggle_key, inputs=mode, outputs=answer_key)

            btn = gr.Button("Evaluate answer sheet →", variant="primary", size="lg")

        # ── RIGHT PANEL ──
        with gr.Column(scale=6, min_width=400):

            gr.HTML('<div class="section-label">Evaluation status</div>')

            status_box = gr.Textbox(
                label="",
                interactive=False,
                placeholder="Status will appear here...",
                elem_classes=["status-box"]
            )

            gr.HTML('<div class="section-label" style="margin-top:16px;">Results</div>')

            result_box = gr.HTML(
                value='<div style="color: var(--text-muted); font-size: 14px; font-family: DM Sans, system-ui, sans-serif; padding: 32px; text-align: center; border: 1px dashed var(--border, #e5e7eb); border-radius: 12px;">Upload answer sheets and click Evaluate to see results</div>',
                elem_classes=["result-area"]
            )

    # ── Wire up ──
    btn.click(
        run_evaluation,
        inputs=[img1, img2, img3, img4, mode, answer_key, policy],
        outputs=[status_box, result_box]
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_9938/1184493300.py:188: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="ExamAI — Answer Sheet Evaluator") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c183f83e2601624eec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
